<table style="width:100%; border-bottom: 2px solid #ccc; margin-bottom: 20px;">
  <tr>
    <td style="vertical-align:middle;">
      <img src="../../resources/ADI-Logo-RGB-FullColor.png" alt="Company Logo" height="30">
    </td>
    <td style="text-align:right; vertical-align:middle;">
      <p style="margin: 0;">Phased Array Systems</p>
      <p style="font-size: 14px; margin: 0;">Iain Derrington – ADEF Group, ADI</p>
      <p style="font-size: 12px; color: #555;">Field Applications & Platform Engineer</p>
    </td>
  </tr>
</table>

In [ ]:
# Common Declarations and setup
sys.path.insert(0, '../src')
import os
import sys
import time

import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from phaser_functions import *
from phaser_init import init_phaser_sdr

from adi import adf4159
from adi import ad9361
from adi import one_bit_adc_dac
from adi import ad9361
from adi import tddn
from adi.cn0566 import CN0566

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output 

from dataclasses import dataclass, fields
from typing import List

# Get Script / Notebook root and full path to resources folder
phaser_root = get_phaser_root()
resource_path = phaser_root / "resources"

display(Markdown(f"Phaser root: **{phaser_root}**"))
display(Markdown(f"Resource path: **{resource_path}**\n"))

# FMCW RADAR: Range-Doppler Processing (2D FFT)

## Overview

In the previous notebook, we successfully measured the **range** of static targets using synchronized FMCW chirps and the Range FFT. However, we had two significant limitations:

1. **No velocity information**: We couldn't determine if a target was moving or how fast
2. **Single chirp processing**: We processed each chirp independently with no coherent integration

This notebook introduces **Range-Doppler processing** using the **2D FFT**, which simultaneously measures both **range** and **velocity** of multiple targets.

<div style="text-align:center;">
  <img src="resources/range_doppler.svg" alt="Phaser Block Diagram" height="300">
</div>

## Learning Objectives

By the end of this notebook, you will:
1. Understand the concept of **fast-time** vs **slow-time** in FMCW RADAR
2. Capture a **Coherent Processing Interval (CPI)** of multiple chirps
3. Implement the **2D FFT** for Range-Doppler processing
4. Generate and interpret **Range-Doppler Maps (RDM)**
5. Extract both range and velocity from moving targets
6. Understand range-velocity ambiguities and coupling
7. Apply MTI (Moving Target Indication) filtering

## Section 1: Fast-Time vs Slow-Time

### The Two Time Dimensions in FMCW

FMCW RADAR operates in **two time domains**:

#### Fast-Time (Within a Chirp)
- Time **within** a single chirp
- Sample rate: $f_s$ (e.g., 600 kHz)
- Used to measure **range** via beat frequency
- FFT along fast-time → Range FFT

#### Slow-Time (Across Chirps)
- Time **between** chirps
- Sample rate: Pulse Repetition Frequency (PRF)
- Used to measure **velocity** via Doppler frequency
- FFT along slow-time → Doppler FFT

### The 2D Data Matrix

When we capture $M$ chirps of $N$ samples each, we build a 2D matrix:

<div style="text-align:center;">
  <img src="resources/range_doppler_columns.svg" alt="Phaser Block Diagram" height="300">
</div>

- **Rows (slow-time)**: Different chirps
- **Columns (fast-time)**: Samples within each chirp

### The 2D FFT Process

1. **Range FFT** (along columns): $N$-point FFT for each chirp → Range profiles
2. **Doppler FFT** (along rows): $M$-point FFT for each range bin → Doppler information

Result: **Range-Doppler Map** (RDM) showing targets in range-velocity space

## Section 2: Doppler Frequency and Velocity

### Doppler Effect Refresher

When a target moves relative to the RADAR, the received frequency shifts:

$$
f_d = \frac{2 v \cdot f_c}{c}
$$

Where:
- $f_d$ = Doppler frequency shift (Hz)
- $v$ = target radial velocity (m/s) (positive = approaching)
- $f_c$ = carrier frequency (Hz)
- $c$ = speed of light (m/s)

### FMCW Doppler: Phase Change Across Chirps

In FMCW, Doppler manifests as a **phase change** in the beat signal from chirp to chirp:

$$
\Delta \phi = 2\pi f_d T_{\text{chirp}}
$$

Where $T_{\text{chirp}}$ is the time between chirp starts (chirp period).

The slow-time FFT detects this phase progression.

### Velocity Resolution

$$
\Delta v = \frac{\lambda}{2 \cdot M \cdot T_{\text{chirp}}}
$$

Where:
- $\lambda$ = wavelength
- $M$ = number of chirps in CPI

**More chirps → better velocity resolution** (but longer CPI duration)

### Maximum Unambiguous Velocity

$$
v_{\text{max}} = \frac{\lambda}{4 \cdot T_{\text{chirp}}} = \frac{\lambda \cdot PRF}{4}
$$

Velocities beyond $v_{\text{max}}$ alias (appear at wrong velocity)

## Section 3: Building the CPI Data Cube

### What is a CPI (Coherent Processing Interval)?

A **CPI** is a sequence of $M$ chirps captured with:
- Fixed chirp parameters (BW, duration, center frequency)
- Consistent timing (same PRF)
- Phase coherence maintained across all chirps

### Data Structure

We'll build a 3D array (data cube):
- **Dimension 0**: Chirp index (slow-time) - size $M$
- **Dimension 1**: Sample index (fast-time) - size $N$
- **Dimension 2**: Channel (RX channel) - size 2 for CN0566

Shape: `(M, N, 2)`

#### Selecting `M` and `N`

When configuring a Range-Doppler radar experiment, two important parameters must be chosen:

- `N`: the number of fast-time samples collected during each chirp
- `M`: the number of chirps captured within one coherent processing interval (CPI)

Together, these parameters define the size of the Range-Doppler data cube and affect range coverage, velocity resolution, processing time, and memory usage.

##### Choosing `N`: Fast-Time Samples

`N` determines how many samples are collected during a single chirp. These samples form the input to the Range FFT, so `N` primarily affects the range axis.

For an indoor demonstration, targets are usually only a few metres from the radar. Since very long measurement ranges are not required, there is little benefit in collecting an excessive number of samples. Instead, choose an `N` that comfortably covers the desired range while keeping acquisition and processing fast.

A larger `N`:

- increases the number of FFT bins in the range dimension
- improves the visual smoothness of the range profile
- increases acquisition and processing time

Typical values are:

| `N` | Use case |
| --- | -------- |
| 512 | Suitable for short-range indoor demonstrations |
| 1024 | Good compromise between range detail and processing cost |
| 2048+ | Usually unnecessary for indoor experiments |

For this tutorial, we use:

```python
N = 1024
```

The PlutoSDR has min sampling frequency of aprox 600ksps. Lets assumen for now we will use an FFT size of 1024 that gives us a frequency resolution of:

$ \large
 \Delta f = \frac {F_s}{N} = \frac {600e3}{1024} = 585.94 Hz 
$  

----

##### Choosing M: Number of Chirps  

M determines how many chirps are captured during one CPI. These chirps form the input to the Doppler FFT, so M primarily affects the velocity axis.  
Velocity is measured by observing phase change from chirp to chirp. Capturing more chirps allows smaller Doppler frequency shifts to be resolved, improving velocity resolution.

A larger M:
- improves velocity resolution
- produces a smoother Doppler spectrum
- increases CPI duration
- makes the display less responsive to rapidly changing scenes

For this indoor demonstration, target velocities are expected to be relatively low, typically hand motion or walking-speed movement. Good Doppler resolution is therefore more useful than supporting very rapid scene updates.
Typical values are:
| M (Number of Chirps) | Use Case |
|----------------------|----------|
| 32 | Fast updates, coarse velocity resolution |
| 64 | Good compromise for indoor demonstrations |
| 128 | Improved Doppler resolution |
| 256 | Very good Doppler resolution, but slower updates |tes

For this tutorial, we use:
```python
M = 64
```

This provides enough Doppler resolution to distinguish stationary objects from slow-moving targets while keeping the display responsive.  

Practical Trade-Off  

The Range-Doppler map is generated from an M x N matrix:
`range_doppler_data.shape = (M, N)`  

Increasing either parameter improves part of the measurement, but also increases computation and memory requirements.  

For an indoor radar demonstration with ranges below approximately 10 m and human-scale motion, a practical starting point is:  

```python
N = 1024  # Samples per chirp
M = 64    # Chirps per CPI   
```   

These values produce a responsive Range-Doppler display while providing enough range and velocity resolution to visualize stationary objects, hand motion, and walking-speed targets.  

Engineering rationale:  
N primarily controls the range axis because it defines how much fast-time data is available for the Range FFT.  
 
M primarily controls the velocity axis because it defines how many chirps are available for the Doppler FFT.

We will want to capture both fast and slow time data in a single buffer.
Thus we need to allocated enough memory

Using the numbers we have select so far:  

$
M \times N =  64 \times 1024 = 65536 \text{ samples}
$

We will need twice this as we have two ADC channels.

What about sample rate. Well lets remind ourselves:

$
\Large f_b = k \frac{2R}{c}
$

where:
- R = range
- k = chirp slope
- c = speed of light

Lets use the slope from the prevous excercise `1.0 THz/s` and assume range will be no more than 10m

$$ \begin{aligned} 
f_b &= 1 \times 10^{12}\,\frac{2 \times 10}{3 \times 10^8} \\ 
    &= 66.7~\text{kHz} 
    \end{aligned} $$

We need sampling frequency 2x 66.7kHz.

Let set fs to 600ksps. This is the lowest frequency that the PlutoSDR supports.

Buffer Size        = 65536 samples  
Chirps per CPI     = 64  
Samples per Chirp  = 1024  

Sample Rate        = 600 kSPS  
Chirp Duration     = 1.707 ms  
Reset Time         = 100 µs  

PRI                = 1.807 ms  
PRF                = 553 Hz  

CPI Duration       = 115.6 ms
Velocity Resolution ≈ 0.13 m/s
`

#### Setup and Imports

In [ ]:
@dataclass
class RadarConfig:
    """Configuration parameters for FMCW radar operation."""

    # ========== SDR Parameters ==========
    sample_rate: float = 0.6e6  # Sample rate

    center_freq: float = 2.1e9  # SDR LO frequency (Hz). Upconverted to output_freq by ADF4159
                                 # Keep at 2.1 GHz for optimal Pluto performance

    signal_freq: float = 100e3  # TX baseband tone frequency (Hz). Creates IF offset
                                 # Used to separate DC offset from target returns
                                 # Typical: 100 kHz. Don't change unless you know why

    rx_gain: int = 60  # Receiver gain (dB). Range: -3 to 70 dB
                        # Higher = more sensitive but risk ADC saturation on strong returns
                        # Start at 60, reduce if seeing saturation artifacts
                        # Trade-off: +10 dB gain ~= 3x detection range OR 10 dB less TX power needed

    tx_gain: int = 0  # Transmitter gain (dB). Range: 0 to -88 dB (0 = max power)
                       # 0 dB ~= +30 dBm EIRP with array gain ~= 1W effective
                       # Reduce for short range, regulations, or power saving
                       # Trade-off: -6 dB power ~= 0.5x detection range

    sdr_buf_size: int = 65536
    fft_size: int = sdr_buf_size

    # ========== Chirp Parameters ==========
    output_freq: float = 9.9e9  # Radar transmit frequency (Hz). X-band (8-12 GHz)
                                 # 9.9 GHz = 30mm wavelength. Good for small targets
                                 # Check local regulations (ISM, amateur, Part 15)

    chirp_BW: float = 500e6  # Chirp bandwidth (Hz). Determines range resolution
                              # Range resolution = c/(2*BW) = 0.3m at 500 MHz
                              # Wider BW = better resolution, more processing
                              # Typical: 250 MHz to 1 GHz (if hardware supports)
                              # Trade-off: 2x BW = 0.5x range resolution (better)

    ramp_time: int = 1700  # Chirp duration (microseconds). Affects max range
                            # Longer = more samples per chirp = better range resolution
                            # Also affects PRF (pulse repetition frequency)
                            # Typical: 100 to 1000 us
                            # Trade-off: 2x ramp_time = 0.5x PRF = 0.5x max unambiguous velocity

    num_chirps: int = 64  # Number of chirps per frame (CPI - Coherent Processing Interval)
                            # More chirps = better Doppler (velocity) resolution
                            # Doppler resolution = lambda/(2*CPI*PRI) where PRI ~= ramp_time
                            # Typical: 64 to 512. Power of 2 for efficient FFT
                            # Trade-off: 2x chirps = 0.5x Doppler resolution, 2x processing time

    # ========== Array Parameters ==========
    element_spacing: float = 0.014  # Antenna element spacing (meters). 14mm ~= lambda/2 at 10 GHz
                                     # lambda/2 spacing prevents grating lobes (spatial aliasing)
                                     # Don't change unless physical array changes

    gain_list: List[int] = None  # Per-element gain (0-127). None = all max (127)
                                  # Can apply taper (Blackman, Taylor) to reduce sidelobes
                                  # Example: [8, 34, 84, 127, 127, 84, 34, 8] for Blackman
                                  # Trade-off: Tapering reduces sidelobes but lowers gain

    # ========== Timing Parameters (Advanced) ==========
    begin_offset_fraction: float = 0.1  # Fraction of chirp to skip at start (0.0-0.3)
                                         # VCO takes time to settle; early samples are non-linear
                                         # 0.1 = skip first 10% of chirp (30 us at 300 us ramp)
                                         # Increase if seeing range artifacts near zero
                                         # Trade-off: More offset = fewer samples = less SNR

    pri_padding_ms: float = 0.1   # Dead time between chirps (milliseconds)
                                  # Allows VCO to reset and prevents chirp overlap
                                  # PRI (Pulse Repetition Interval) = ramp_time + padding
                                  # Affects PRF and max unambiguous velocity
                                  # Trade-off: More padding = lower PRF = lower max velocity

    # ========== TDD (Time Division Duplex) Parameters ==========
    tdd_trigger_on_raw: int = 0   # TDD GPIO trigger start (raw units)
                                   # Synchronizes chirp generation with data capture
                                   # Keep at 0 for immediate trigger

    tdd_trigger_off_raw: int = 20  # TDD GPIO trigger stop (raw units)
                                    # Pulse width for trigger signal
                                    # Typical: 5-20. Must be long enough for hardware to latch
                                    # Trade-off: Longer pulse more reliable but delays start

    rpi_ip:       str = "192.168.1.10"
    sdr_ip:       str = "192.168.2.1"
    fieldfox_ip:  str = "192.168.1.30"

    def __post_init__(self):
        """Set default gain list and calibration file paths if not provided."""
        if self.gain_list is None:
            self.gain_list = [127] * 8

    def __iter__(self):
        for field in fields(self):
            yield field.name, getattr(self, field.name)

#  Default are stored in a dataclass
config = RadarConfig()

display(Markdown("#### Config values"))
for name, value in config:    
    display(Markdown(f"- {name} = {value}"))

pll    = None      
gpio   = None
tdd    = None
phaser = None
sdr    = None


### Hardware Configuration

Configure hardware, this is very similar to the previous notebook.
The main differences are:

- TDD is configured for 64 rising edges, each one separated by 1800uS
- ADF4159 ramp time is longer, that previous example.
- Tx/Rx Buffer is larger 65k

In [ ]:
def connect_devices():
    try:
        global pll, gpio, tdd, phaser, sdr
        display(Markdown(f"- ADF4159: ip: {config.rpi_ip}"))
        pll    = adf4159        (uri="ip:" + config.rpi_ip)
        
        display(Markdown(f"- GPIO: ip: {config.rpi_ip}"))
        gpio   = one_bit_adc_dac(uri="ip:" + config.rpi_ip)

        display(Markdown(f"- TDDN: ip: {config.sdr_ip}"))
        tdd = tddn(uri="ip:" + config.sdr_ip)

        display(Markdown(f"- Phaser: ip: {config.rpi_ip}"))
        phaser = CN0566         (uri="ip:" + config.rpi_ip)

        display(Markdown(f"- SDR: ip: {config.rpi_ip}"))
        sdr    = ad9361         (uri="ip:" + config.sdr_ip)
    
    except:
        display(Markdown(f"Unable to connect to CN0566 / ADF4159. Please check the IP addresses and connections."))
        print("")
        sys.exit(1)
    
    # Set phaser.sdr to instance of PlutoSDR
    phaser.sdr = sdr 

def configure_phaser():
    phaser.configure(device_mode="rx")
    phaser.element_spacing = config.element_spacing
    
    for i in range(0, 8):
        phaser.set_chan_phase(i, 0)

    display(Markdown(f"- Phaser channel phase  = 0"))
    
    for i in range(0, len(config.gain_list)):
        phaser.set_chan_gain(i, config.gain_list[i], apply_cal=False)
    
    phaser._gpios.gpio_tx_sw = 0
    phaser._gpios.gpio_vctrl_1 = 1
    phaser._gpios.gpio_vctrl_2 = 1

def configure_sdr():   
    destroy_sdr_buffer()
    
    # Configure sample rate to capture the full frame (chirp + padding)
    # Frame time = ramp_time + pri_padding_ms
    frame_time_s = (config.ramp_time * 1e-6) + (config.pri_padding_ms * 1e-3)
    phaser.sdr.sample_rate = int(config.sample_rate)
    
    phaser.sdr.rx_lo = int(config.center_freq)
    phaser.sdr.rx_enabled_channels = [0, 1]
    phaser.sdr.rx_buffer_size = config.sdr_buf_size
    
    phaser.sdr.gain_control_mode_chan0 = 'manual'
    phaser.sdr.gain_control_mode_chan1 = 'manual'
    phaser.sdr.rx_hardwaregain_chan0 = -88
    phaser.sdr.rx_hardwaregain_chan1 = config.rx_gain

    phaser.sdr.tx_buffer_size = config.sdr_buf_size
    phaser.sdr.tx_lo = int(config.center_freq)
    phaser.sdr.tx_enabled_channels = [0, 1]
    phaser.sdr.tx_cyclic_buffer = True
    phaser.sdr.tx_hardwaregain_chan0 = -88
    phaser.sdr.tx_hardwaregain_chan1 = int(config.tx_gain)

    display(Markdown(f"- Sample rate: {sdr.sample_rate/1e6:.2f} MHz"))
    display(Markdown(f"- Frame time: {frame_time_s*1e3:.2f} ms (chirp + padding)"))
    display(Markdown(f"- Chirp time: {config.ramp_time} us"))
    display(Markdown(f"- Padding time: {config.pri_padding_ms} ms"))
    display(Markdown(f"- RX LO: {sdr.rx_lo/1e9:.1f} GHz"))
    display(Markdown(f"- TX LO: {sdr.tx_lo/1e9:.1f} GHz"))
    display(Markdown(f"- Buffer size: {sdr.rx_buffer_size} samples"))
    display(Markdown(f"- Capture time: {sdr.rx_buffer_size/sdr.sample_rate*1e3:.2f} ms"))

def destroy_sdr_buffer():
    try: sdr.tx_destroy_buffer()
    except: pass
    try: sdr.rx_destroy_buffer()
    except: pass

def configure_adf4159():
    # Configure ADF4159 for triggered sawtooth chirp

    vco_freq = int(config.output_freq + config.signal_freq + config.center_freq)
    BW = config.chirp_BW
    num_steps = int(config.ramp_time)

    display(Markdown(f"- VCO Output Frequency = {vco_freq/1e9} GHz"))

    phaser.frequency = int(vco_freq / 4)
    phaser.freq_dev_range = int(BW / 4)
    phaser.freq_dev_step = int((BW / 4) / num_steps)
    phaser.freq_dev_time = int(config.ramp_time)
    
    phaser.delay_word = 4095
    phaser.delay_clk = "PFD"
    phaser.delay_start_en = 0
    phaser.ramp_delay_en = 0
    phaser.trig_delay_en = 0
    phaser.ramp_mode = "single_sawtooth_burst"
    phaser.sing_ful_tri = 0
    phaser.tx_trig_en = 1
    phaser.enable = 0                                  # 0 = PLL enable.  Write this last to update all the registers

    display(Markdown(f"- Chirp BW: {(4 * phaser.freq_dev_range)/1e6:.0f} MHz"))
    display(Markdown(f"- Ramp time: {config.ramp_time} us"))
    display(Markdown(f"- Chirp rate: {config.chirp_BW/(config.ramp_time*1e-6)/1e12:.2f} THz/s"))

def configure_tdd():
    """
    Configure the TDD (Time Division Duplex) controller so that each FMCW chirp
    is synchronised to the data acquisition hardware.

    The TDD engine generates trigger signals at the start of each chirp period
    (PRI) and repeats this for the required number of chirps in a burst.
    """

    # Route the TDD trigger to the external sync circuitry and enable
    # the Phaser board trigger path.
    gpio.gpio_tdd_ext_sync = True
    gpio.gpio_phaser_enable = True

    # Disable the TDD engine while its configuration is updated.
    tdd.enable = False

    # Use an external trigger source to start the TDD sequence.
    tdd.sync_external = True

    # Begin generating triggers immediately after synchronisation.
    tdd.startup_delay_ms = 0

    # Calculate the Pulse Repetition Interval (PRI).
    #
    # The PRI consists of:
    #   - the FMCW ramp (chirp) duration
    #   - additional padding time between chirps
    #
    # ramp_time is stored in microseconds, so convert to milliseconds
    # before adding the padding value.
    PRI_ms = config.ramp_time / 1e3 + config.pri_padding_ms

    
    # Set the time between successive chirp triggers.
    tdd.frame_length_ms = PRI_ms

    display(Markdown(f"- Frame Len =  {tdd.frame_length_ms} ms"))

    # Generate one trigger event per chirp in the burst.
    tdd.burst_count = config.num_chirps

    display(Markdown(f"- No. Chirps / trigger =  {config.num_chirps}"))

    tdd.channel[0].enable = True
    tdd.channel[0].polarity = False
    tdd.channel[0].on_raw = config.tdd_trigger_on_raw
    tdd.channel[0].off_raw = config.tdd_trigger_off_raw

    tdd.channel[1].enable = True
    tdd.channel[1].polarity = False
    tdd.channel[1].on_raw = config.tdd_trigger_on_raw
    tdd.channel[1].off_raw = config.tdd_trigger_off_raw
    
    tdd.channel[2].enable = True
    tdd.channel[2].polarity = False
    tdd.channel[2].on_raw = config.tdd_trigger_on_raw
    tdd.channel[2].off_raw = config.tdd_trigger_off_raw
    
    # Apply the configuration and start the TDD engine.
    tdd.enable = True

def tx_baseband():
    # Generate baseband transmit waveform (tone at IF)

    fs = int(sdr.sample_rate)
    N = config.sdr_buf_size
    t = np.arange(N) / fs

    i_data = np.cos(2 * np.pi * t * config.signal_freq) * 2**14
    q_data = np.sin(2 * np.pi * t * config.signal_freq) * 2**14
    iq_data = i_data + 1j * q_data

    display(Markdown(f"- Transmit {config.signal_freq/1e3} kHz"))
    display(Markdown(f"- TX buffer size: {len(iq_data)} samples"))

    # Send waveform to TX buffer
    sdr.tx([iq_data, iq_data])

display(Markdown("**Connect to Devices**"))
connect_devices()

display(Markdown("**Configure Phaser**"))
configure_phaser()

display(Markdown("**Configuring PlutoSDR**"))
destroy_sdr_buffer()
configure_sdr()

display(Markdown("**Configuring ADF4159**"))
configure_adf4159()    

display(Markdown("**Configure TDD Engine**"))
configure_tdd()      

display(Markdown("**Transmit Baseband Signal**"))
tx_baseband()

### CPI Capture Function

TODO: Implement CPI data capture

In [ ]:
def capture_data():
    """
    Trigger TDD and capture synchronized data
    """
    # Trigger TDD sequence (starts one chirp)
    phaser._gpios.gpio_burst = 0
    phaser._gpios.gpio_burst = 1
    phaser._gpios.gpio_burst = 0

    time.sleep(0.001)

    # In this example the rxdata should contain 64 reflected chirp signals
    rx_data = sdr.rx()

    return rx_data

def process_data(raw_data):
    """
    Process raw data
    This is where we can do monopulse
    raw_data is a list of two channels: [ch0, ch1]
    """
    rx_ch = raw_data[0] + raw_data[1]

    return rx_ch

    
# Live beat frequency and range display with spectrogram
print("Starting live FMCW capture...")
print("Press Ctrl+C (Interrupt kernel) to stop")
print()
print(f"Expected beat frequency for 2m target: ~13.3 kHz")
print(f"Range resolution: {3e8/(2*config.chirp_BW):.2f} m")
print()

actual_sample_rate = sdr.sample_rate
chirp_samples = int(actual_sample_rate * config.ramp_time * 1e-6)

# Spectrogram history buffer
HISTORY_LENGTH = 100  # Keep last 100 frames
spectrogram_history = []

# Create figure and axes ONCE
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 14))

# Initialize plot elements that will be updated
line1_i, = ax1.plot([], [], 'b-', linewidth=0.8, alpha=0.6, label='I')
line1_q, = ax1.plot([], [], 'r-', linewidth=0.8, alpha=0.6, label='Q')
line1_mag, = ax1.plot([], [], 'g-', linewidth=1.2, label='Mag')
line2_fft, = ax2.plot([], [], 'b-', linewidth=1)
vline1 = None
vspan1 = None
im3 = None
cbar3 = None

ax1.set_xlabel('Time (ms)', fontsize=11)
ax1.set_ylabel('Amplitude', fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.legend(loc='upper right', fontsize=9)

ax2.set_xlabel('Beat Frequency (kHz)', fontsize=11)
ax2.set_ylabel('Magnitude (dB)', fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.set_xlim([60, 140])

ax3.set_xlabel('Beat Frequency (kHz)', fontsize=11)
ax3.set_ylabel('Frame Number', fontsize=11)
ax3.grid(True, alpha=0.3, color='white', linewidth=0.5)

plt.tight_layout()

try:
    frame_count = 0

    while True:
        frame_count += 1

        raw_data = capture_data()
        rx_data = process_data(raw_data)

        # Only process the ramp time...
        rx_chirp = rx_data[:chirp_samples]

        # === Update Plot 1: Time Domain ===
        t_ms = np.arange(len(rx_data)) / actual_sample_rate * 1e3
        i_signal = np.real(rx_data)
        q_signal = np.imag(rx_data)
        magnitude = np.abs(rx_data)

        line1_i.set_data(t_ms, i_signal)
        line1_q.set_data(t_ms, q_signal)
        line1_mag.set_data(t_ms, magnitude)
        
        ax1.set_xlim([0, t_ms[-1]])
        ax1.set_ylim([np.min(i_signal)*1.1, np.max(magnitude)*1.1])
        ax1.set_title(f'De-chirped Signal - Frame {frame_count}', fontsize=13, fontweight='bold')
        
        # Update chirp end marker
        if vline1:
            vline1.remove()
        if vspan1:
            vspan1.remove()
        chirp_end_time = config.ramp_time / 1e3
        vline1 = ax1.axvline(chirp_end_time, color='orange', linestyle='--', linewidth=2, alpha=0.7)
        vspan1 = ax1.axvspan(chirp_end_time, t_ms[-1], alpha=0.2, color='red')

        # === FFT Processing ===
        win = np.blackman(len(rx_chirp))
        windowed = rx_chirp * win
        spectrum = np.fft.fft(windowed, n=config.fft_size)
        spectrum_pos = spectrum[:config.fft_size//2]
        freqs_pos = np.fft.fftfreq(config.fft_size, 1/actual_sample_rate)[:config.fft_size//2]
        magnitude_db = 20 * np.log10(np.abs(spectrum_pos) + 1e-12)

        chirp_time_s = config.ramp_time * 1e-6
        chirp_rate = config.chirp_BW / chirp_time_s
        IF_freq = config.signal_freq
        beat_freq = np.abs(freqs_pos - IF_freq)
        range_bins_m = (3e8 * beat_freq) / (2 * chirp_rate)

        spectrogram_history.append(magnitude_db.copy())
        if len(spectrogram_history) > HISTORY_LENGTH:
            spectrogram_history.pop(0)

        # === Update Plot 2: FFT ===
        line2_fft.set_data(freqs_pos / 1e3, magnitude_db)
        ax2.set_ylim([np.max(magnitude_db)-60, np.max(magnitude_db)+5])
        ax2.set_title(f'FFT Spectrum (Chirp: {chirp_samples} samples)', fontsize=13, fontweight='bold')

        # Clear old peak annotations and redraw
        for txt in ax2.texts:
            txt.remove()
        for artist in ax2.artists:
            artist.remove()
        for line in ax2.lines[1:]:  # Keep first line (FFT), remove peak markers
            line.remove()

        threshold = np.max(magnitude_db) - 15
        peak_indices = []
        for i in range(10, len(magnitude_db)-10):
            if magnitude_db[i] > threshold:
                if magnitude_db[i] > magnitude_db[i-1] and magnitude_db[i] > magnitude_db[i+1]:
                    peak_indices.append(i)

        for i, peak_idx in enumerate(peak_indices[:3]):
            peak_freq = freqs_pos[peak_idx] / 1e3
            peak_range = range_bins_m[peak_idx]
            peak_mag = magnitude_db[peak_idx]
            ax2.plot(peak_freq, peak_mag, 'ro', markersize=10)
            ax2.annotate(f'{peak_freq:.1f} kHz\n{peak_range:.2f} m',
                        xy=(peak_freq, peak_mag), xytext=(10, -20),
                        textcoords='offset points', fontsize=9, ha='left',
                        bbox=dict(boxstyle='round,pad=0.4', fc='yellow', alpha=0.7),
                        arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0.2'))

        # === Update Plot 3: Spectrogram ===
        if len(spectrogram_history) > 1:
            spectrogram_data = np.array(spectrogram_history)
            freq_min_idx = np.argmin(np.abs(freqs_pos / 1e3 - 60))
            freq_max_idx = np.argmin(np.abs(freqs_pos / 1e3 - 140))
            spectrogram_subset = spectrogram_data[:, freq_min_idx:freq_max_idx]
            freqs_subset = freqs_pos[freq_min_idx:freq_max_idx] / 1e3

            if im3 is None:
                # First time: create image and colorbar with constrained width
                extent = [freqs_subset[0], freqs_subset[-1], 0, len(spectrogram_history)]
                im3 = ax3.imshow(spectrogram_subset, aspect='auto', origin='lower',
                               extent=extent, cmap='viridis', interpolation='bilinear',
                               vmin=np.median(spectrogram_subset),
                               vmax=np.max(spectrogram_subset))
                # Create colorbar with specific width so it doesn't shrink axes
                from mpl_toolkits.axes_grid1 import make_axes_locatable
                divider = make_axes_locatable(ax3)
                cax = divider.append_axes("right", size="2%", pad=0.1)
                cbar3 = plt.colorbar(im3, cax=cax)
                cbar3.set_label('Magnitude (dB)', fontsize=10)
            else:
                # Update existing image
                extent = [freqs_subset[0], freqs_subset[-1], 0, len(spectrogram_history)]
                im3.set_data(spectrogram_subset)
                im3.set_extent(extent)
                im3.set_clim(vmin=np.median(spectrogram_subset), vmax=np.max(spectrogram_subset))

            ax3.set_title(f'Range-Time Spectrogram (Last {len(spectrogram_history)} frames)',
                         fontsize=13, fontweight='bold')

        fig.canvas.draw()
        fig.canvas.flush_events()
        clear_output(wait=True)
        display(fig)
        plt.pause(0.01)

except KeyboardInterrupt:
    print("\nLive capture stopped")
    print(f"  Total frames captured: {frame_count}")

plt.close()


## Section 4: 2D FFT Implementation

### Step-by-Step 2D FFT Processing

1. **Range FFT** (1st dimension):
   - Apply window along fast-time (per chirp)
   - $N$-point FFT for each chirp
   - Result: Range profiles for each chirp

2. **Doppler FFT** (2nd dimension):
   - Apply window along slow-time (per range bin)
   - $M$-point FFT for each range bin
   - Result: Range-Doppler Map

### Windowing in Both Dimensions

- **Range dimension**: Reduces range sidelobes (e.g., Blackman window)
- **Doppler dimension**: Reduces velocity sidelobes (e.g., Hann window)

Trade-off: Resolution vs sidelobe suppression

TODO: Implement 2D FFT processing

In [ ]:
# TODO: 2D FFT function
def range_doppler_fft(data_cube, range_window='blackman', doppler_window='hann'):
    """
    Perform 2D FFT to generate Range-Doppler Map
    
    Parameters:
    -----------
    data_cube : ndarray
        Shape (M, N, num_channels)
    range_window : str
        Window for range dimension
    doppler_window : str
        Window for Doppler dimension
    
    Returns:
    --------
    rdm : ndarray
        Range-Doppler Map (magnitude, dB)
    range_bins : ndarray
        Range values (m)
    doppler_bins : ndarray
        Doppler frequencies (Hz)
    velocity_bins : ndarray
        Velocity values (m/s)
    """
    # TODO: Implement
    # 1. Apply range window and FFT along axis 1
    # 2. Apply Doppler window and FFT along axis 0
    # 3. FFT shift to center zero Doppler
    # 4. Convert to magnitude (dB)
    # 5. Calculate axes
    pass

## Section 5: Range-Doppler Map (RDM) Visualization

### Understanding the RDM

The Range-Doppler Map is a 2D image where:
- **X-axis**: Velocity (or Doppler frequency)
- **Y-axis**: Range
- **Color/Intensity**: Signal power (dB)

Each **bright spot** represents a target at a specific range and velocity.

### RDM Features to Look For

- **Zero Doppler column**: Static clutter (ground, walls, stationary objects)
- **Off-zero Doppler peaks**: Moving targets
- **Positive Doppler**: Targets approaching (coming toward RADAR)
- **Negative Doppler**: Targets receding (moving away)

TODO: Create RDM visualization function

In [ ]:
# TODO: RDM visualization
def plot_rdm(rdm, range_bins, velocity_bins, vmin=-60, vmax=0):
    """
    Plot Range-Doppler Map
    
    Parameters:
    -----------
    rdm : ndarray
        Range-Doppler map (dB)
    range_bins : ndarray
        Range axis values
    velocity_bins : ndarray
        Velocity axis values
    vmin, vmax : float
        Color scale limits (dB)
    """
    # TODO: Implement
    # - Create 2D image plot
    # - Label axes properly
    # - Add colorbar
    # - Mark zero Doppler line
    pass

### Single CPI Processing and Display

TODO: Capture, process, and display one RDM

In [ ]:
# TODO: Single RDM demo
# - Capture CPI
# - Process 2D FFT
# - Display RDM
# - Interpret results

## Section 6: Target Detection in Range-Doppler Space

### 2D Peak Detection

To extract target parameters:
1. Apply threshold to RDM
2. Find local maxima (peaks)
3. For each peak:
   - Range bin → range
   - Doppler bin → velocity
   - Peak magnitude → RCS/SNR

TODO: Implement 2D peak detection

In [ ]:
# TODO: 2D peak detection
def detect_targets_2d(rdm, range_bins, velocity_bins, threshold_db=-20):
    """
    Detect targets in Range-Doppler Map
    
    Parameters:
    -----------
    rdm : ndarray
        Range-Doppler map (dB)
    range_bins : ndarray
        Range values
    velocity_bins : ndarray
        Velocity values
    threshold_db : float
        Detection threshold (dB below peak)
    
    Returns:
    --------
    detections : list of dict
        Each detection: {'range': R, 'velocity': v, 'snr': SNR}
    """
    # TODO: Implement
    # 1. Find peaks above threshold
    # 2. Extract range and velocity for each
    # 3. Calculate SNR
    pass

## Section 7: MTI (Moving Target Indication) Filtering

### The Static Clutter Problem

In many scenarios, **static clutter** (walls, ground, stationary objects) dominates the RDM at **zero Doppler**, masking weak moving targets.

### MTI: High-Pass Filter in Slow-Time

MTI removes the zero-Doppler component by:
1. Subtracting mean across chirps (simple MTI)
2. Or applying high-pass filter along slow-time

$$
\text{MTI: } x_{\text{MTI}}[m, n] = x[m, n] - \frac{1}{M}\sum_{m=0}^{M-1} x[m, n]
$$

This **removes DC** in the slow-time domain → removes static clutter.

### Higher-Order MTI

For better clutter rejection:
- **2-pulse canceller**: $y[m] = x[m] - x[m-1]$
- **3-pulse canceller**: $y[m] = x[m] - 2x[m-1] + x[m-2]$

TODO: Implement MTI filtering

In [ ]:
# TODO: MTI filter
def apply_mti(data_cube, order=1):
    """
    Apply Moving Target Indication filter
    
    Parameters:
    -----------
    data_cube : ndarray
        Shape (M, N, channels)
    order : int
        MTI filter order (1=simple mean removal, 2=2-pulse canceller)
    
    Returns:
    --------
    data_mti : ndarray
        Clutter-suppressed data
    """
    # TODO: Implement
    # - Subtract mean across chirps (axis 0)
    # - Or apply differencing filter
    pass

### Comparison: With and Without MTI

TODO: Show side-by-side RDMs with/without MTI

In [ ]:
# TODO: MTI comparison
# - Process RDM without MTI
# - Process RDM with MTI
# - Plot side-by-side
# - Highlight moving target detection improvement

## Section 8: Range-Velocity Coupling

### The Coupling Problem

In triangular FMCW, there's a **coupling** between measured range and velocity:

- **Up-chirp**: $f_{\text{beat}} = f_R + f_D$
- **Down-chirp**: $f_{\text{beat}} = f_R - f_D$

Where:
- $f_R$ = range-induced beat frequency
- $f_D$ = Doppler shift

If we only use **up-chirps** (or only down-chirps), we can't separate $f_R$ and $f_D$.

### Solution: Up-Down Chirp Pairs

By processing both up and down chirps:

$$
f_R = \frac{f_{\text{up}} + f_{\text{down}}}{2}
$$

$$
f_D = \frac{f_{\text{up}} - f_{\text{down}}}{2}
$$

This **decouples** range from velocity.

### Implementation Note

For the current TDD system with sawtooth chirps:
- Coupling is small for low velocities
- For high-precision applications, implement triangular chirps and decouple

TODO: Demonstrate coupling effect

In [ ]:
# TODO: Range-velocity coupling demo
# - Simulate moving target at known range/velocity
# - Show apparent range shift due to Doppler
# - Calculate coupling error

## Section 9: Live Range-Doppler Demonstration

### Interactive RDM Display

TODO: Create live updating Range-Doppler map

In [ ]:
# TODO: Live RDM demo
# - Continuous CPI capture
# - Real-time 2D FFT processing
# - Update RDM display
# - Overlay detected targets
# - Display target list (range, velocity, SNR)
# - User can move target and observe changes

### Experiment Ideas

Try these experiments:

1. **Static target**: Place target, observe at zero Doppler
2. **Walking target**: Have someone walk toward/away from RADAR
3. **Hand waving**: Observe micro-Doppler from hand motion
4. **Multiple targets**: Have multiple people moving at different velocities
5. **MTI effectiveness**: Compare static clutter rejection with/without MTI

## Section 10: Summary and Next Steps

### What We've Accomplished

In this notebook, we:
1. ✓ Understood fast-time vs slow-time dimensions
2. ✓ Captured coherent multi-chirp CPIs
3. ✓ Implemented the 2D FFT (Range-Doppler processing)
4. ✓ Generated Range-Doppler Maps
5. ✓ Detected and tracked moving targets
6. ✓ Applied MTI filtering for clutter rejection
7. ✓ Measured both range AND velocity simultaneously

### Key Insights

- **2D FFT** provides simultaneous range-velocity measurement
- **More chirps** → better velocity resolution (but longer CPI)
- **MTI filtering** is essential for moving target detection in cluttered environments
- **Range-velocity coupling** exists but can be mitigated

### Current Capabilities

We can now:
- Detect multiple targets
- Measure range and velocity
- Reject static clutter
- Track moving objects

### Next Notebook: Beamforming Integration

In the next notebook (`6_FMCW_Beamforming_Integration.ipynb`), we'll:
- Add **angle estimation** using the phased array
- Implement **3D tracking**: Range + Velocity + Angle
- Use beam steering to focus on specific targets
- Create **Range-Angle-Doppler cubes**
- Demonstrate full RADAR tracking capabilities

This brings together everything: beamforming + FMCW + Doppler processing!

## Appendix A: 2D FFT Mathematics

### Discrete 2D FFT

For data matrix $x[m, n]$ of size $M \times N$:

$$
X[k, l] = \sum_{m=0}^{M-1} \sum_{n=0}^{N-1} x[m, n] \cdot e^{-j2\pi\left(\frac{km}{M} + \frac{ln}{N}\right)}
$$

### Separable Transform

The 2D FFT can be computed as two sequential 1D FFTs:

1. $Y[m, l] = \text{FFT}_N\{x[m, n]\}$ (FFT along each row)
2. $X[k, l] = \text{FFT}_M\{Y[m, l]\}$ (FFT along each column)

### Computational Complexity

- 1D FFT: $O(N \log N)$
- 2D FFT: $O(MN(\log M + \log N))$

For typical values ($M=64$, $N=512$):
- Total operations: ~$3 \times 10^5$ complex multiplies
- Real-time processing easily achievable on modern CPUs

## Appendix B: Doppler Ambiguity Resolution

### PRF Selection Trade-offs

| PRF | Advantage | Disadvantage |
|-----|-----------|-------------|
| High PRF | Large $v_{\text{max}}$ | Small $R_{\text{max}}$ |
| Low PRF | Large $R_{\text{max}}$ | Small $v_{\text{max}}$ |
| Medium PRF | Balanced | Ambiguities in both |

### Multiple PRF Technique

Use two or more PRFs and resolve ambiguities using Chinese Remainder Theorem:
1. Capture CPI at PRF₁
2. Capture CPI at PRF₂
3. Resolve velocity ambiguity algebraically

Extended unambiguous velocity:
$$
v_{\text{max,extended}} = v_{\text{max,1}} \times v_{\text{max,2}} / \gcd(v_{\text{max,1}}, v_{\text{max,2}})
$$